In [72]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas import DataFrame

In [ ]:
data = pd.read_csv("../data/final_data.csv")
data.head()

,date,symbol,sector,entry,sl,t1,t2,exit_price,outcome,return_pct,...,Vol_ratio,Dist_from_200MA,sma50_above_sma200,perfect_bullish_alignment,price_above_ema9,price_above_sma50,rev_pat,cont_pat,candle_pat,dow_signal_encoded
0,2022-07-08,TIMETECHNO,Consumer Cyclical,53.80,49.50,66.72,79.63,49.50,-1,-8.00,...,0.61,34.66,1,1,1,1,0.00,0.85,-0.8,0
1,2022-07-15,TIMETECHNO,Consumer Cyclical,53.07,48.83,65.81,78.55,48.83,-1,-8.00,...,0.64,31.24,1,1,0,1,0.00,-1.00,0.0,1
2,2022-07-22,TIMETECHNO,Consumer Cyclical,49.74,48.99,51.98,54.22,48.99,-1,-1.50,...,0.77,21.63,1,0,0,0,-0.65,-1.00,-0.8,1
3,2022-07-29,TIMETECHNO,Consumer Cyclical,52.10,47.93,64.60,77.11,57.14,1,9.67,...,1.16,26.20,1,0,1,1,-0.65,-1.00,0.0,0
4,2022-08-05,TIMETECHNO,Consumer Cyclical,51.69,47.70,63.65,75.61,53.77,1,4.03,...,0.68,24.06,1,0,1,1,-0.65,1.00,0.0,0


In [74]:
data.columns
data['date'] = pd.to_datetime(data['date'])

In [75]:
grade_map = {
    'A+' : 5,
    'A' : 4,
    'B':3,
    'C':2,
    'D':1 ,
    'F':0
    
}

data['grade'] = data['grade'].map(grade_map)


In [76]:
data['grade'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 94313 entries, 0 to 94312
Series name: grade
Non-Null Count  Dtype
--------------  -----
94313 non-null  int64
dtypes: int64(1)
memory usage: 736.9 KB


In [77]:
data = data.sort_values("date").reset_index(drop=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94313 entries, 0 to 94312
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       94313 non-null  datetime64[ns]
 1   symbol                     94313 non-null  object        
 2   sector                     94313 non-null  object        
 3   entry                      94313 non-null  float64       
 4   sl                         94313 non-null  float64       
 5   t1                         94313 non-null  float64       
 6   t2                         94313 non-null  float64       
 7   exit_price                 94313 non-null  float64       
 8   outcome                    94313 non-null  int64         
 9   return_pct                 94313 non-null  float64       
 10  rel_return_pct             94313 non-null  float64       
 11  held_days                  94313 non-null  int64         
 12  grad

In [78]:
train_cutoff = (int)(len(data)*.70)
cutoff_date = data.loc[train_cutoff, 'date']

# skip 45 days and then start test data  from it
embargo_days = 45
test_start = cutoff_date + pd.Timedelta(days= embargo_days)
test_start
 


Timestamp('2025-06-22 00:00:00')

In [79]:
test_df = data[data['date'] >= test_start]
test_df.head()

,date,symbol,sector,entry,sl,t1,t2,exit_price,outcome,return_pct,...,Vol_ratio,Dist_from_200MA,sma50_above_sma200,perfect_bullish_alignment,price_above_ema9,price_above_sma50,rev_pat,cont_pat,candle_pat,dow_signal_encoded
69158,2025-06-23,DJML,Industrials,100.71,99.20,105.24,109.78,99.20,-1,-1.50,...,2.08,-23.16,0,0,0,0,0.0,-1.0,-0.80,0
69159,2025-06-23,AEGISLOG,Energy,777.35,759.28,831.54,885.73,759.28,-1,-2.32,...,0.47,0.05,1,0,0,0,0.0,-1.0,-0.80,0
69160,2025-06-23,DATAPATTNS,Industrials,3014.66,2817.35,3606.58,4198.50,2817.35,-1,-6.54,...,0.59,31.69,1,0,1,1,0.0,-1.0,0.55,1
69161,2025-06-23,GRINDWELL,Industrials,1619.39,1572.97,1758.66,1897.92,1572.97,-1,-2.87,...,0.49,-15.72,0,0,0,0,0.3,-1.0,-0.80,-1
69162,2025-06-23,INDIAMART,Communication Services,2440.72,2315.58,2816.13,3191.54,2454.23,1,0.55,...,0.73,7.41,0,0,1,1,-0.5,1.0,-0.80,-1


In [80]:
train_df = data[:train_cutoff]
train_df.tail()

,date,symbol,sector,entry,sl,t1,t2,exit_price,outcome,return_pct,...,Vol_ratio,Dist_from_200MA,sma50_above_sma200,perfect_bullish_alignment,price_above_ema9,price_above_sma50,rev_pat,cont_pat,candle_pat,dow_signal_encoded
66014,2025-05-08,BHEL,Industrials,216.71,211.89,231.20,245.68,211.89,-1,-2.23,...,0.63,-9.98,0,0,0,1,0.85,-1.00,0.80,-1
66015,2025-05-08,VARDHACRLC,Consumer Cyclical,40.25,37.96,47.11,53.97,41.12,2,2.16,...,2.25,-20.57,0,0,0,0,-0.60,1.00,-0.80,-1
66016,2025-05-08,CCL,Consumer Defensive,721.63,663.90,894.82,1068.02,789.30,2,9.38,...,3.43,9.23,0,0,1,1,0.65,0.90,-0.55,2
66017,2025-05-08,CELEBRITY,Consumer Cyclical,11.54,11.11,12.82,14.10,11.11,-1,-3.69,...,0.30,-21.60,0,0,0,1,0.30,0.90,-0.80,-1
66018,2025-05-08,ALICON,Industrials,661.61,630.62,754.60,847.59,847.59,3,28.11,...,0.59,-37.67,0,0,0,0,-1.00,-0.85,0.00,-1


In [81]:
excluded_cols = ["date", "symbol",  "sector" ,                            
    "entry", "sl", "t1", "t2", "exit_price", "outcome",  
    "return_pct", "rel_return_pct", "held_days",
    "mfe_pct", "mae_pct", "trade_win",                 
     ]


In [82]:
train_df = train_df.drop(columns=excluded_cols )
train_df

,grade,model_win,RSI_signal,MACD,MACD_signal,MACD_hist,MACD_cross,BB_width,BB_pct,ATR_pct,Vol_ratio,Dist_from_200MA,sma50_above_sma200,perfect_bullish_alignment,price_above_ema9,price_above_sma50,rev_pat,cont_pat,candle_pat,dow_signal_encoded
0,1,1,2,1.12,0.85,0.27,0,16.84,0.94,5.11,0.61,34.66,1,1,1,1,0.00,0.85,-0.80,0
1,2,1,2,4.75,3.75,1.01,0,24.65,0.86,4.08,0.80,4.29,0,0,1,1,-1.00,1.00,0.55,-1
2,1,1,2,-0.27,-3.10,2.84,0,32.27,0.83,6.23,0.87,34.58,1,0,1,1,0.00,-1.00,0.55,0
3,1,1,2,2.50,6.50,-4.00,0,4.02,0.45,2.66,0.70,-3.22,0,0,1,1,-0.75,0.90,0.00,-1
4,1,1,2,9.90,4.50,5.40,0,14.87,0.92,3.12,2.64,1.03,0,0,1,1,0.50,1.00,0.80,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66014,1,0,-2,3.30,4.92,-1.63,0,12.90,0.30,4.01,0.63,-9.98,0,0,0,1,0.85,-1.00,0.80,-1
66015,1,1,-2,-0.30,-0.10,-0.20,0,13.91,0.41,4.01,2.25,-20.57,0,0,0,0,-0.60,1.00,-0.80,-1
66016,3,1,2,28.37,14.79,13.57,0,32.78,1.00,5.13,3.43,9.23,0,0,1,1,0.65,0.90,-0.55,2
66017,2,0,-2,0.10,0.10,-0.00,-1,17.56,0.47,7.16,0.30,-21.60,0,0,0,1,0.30,0.90,-0.80,-1


In [83]:
train_df.to_csv("./data/train_data.csv" , index= False)


In [84]:
test_df = test_df.drop(columns=excluded_cols )
test_df

,grade,model_win,RSI_signal,MACD,MACD_signal,MACD_hist,MACD_cross,BB_width,BB_pct,ATR_pct,Vol_ratio,Dist_from_200MA,sma50_above_sma200,perfect_bullish_alignment,price_above_ema9,price_above_sma50,rev_pat,cont_pat,candle_pat,dow_signal_encoded
69158,1,0,-2,-3.58,-2.80,-0.79,0,16.78,0.06,5.13,2.08,-23.16,0,0,0,0,0.00,-1.0,-0.80,0
69159,2,1,-2,-7.08,-5.69,-1.40,0,4.85,0.27,3.31,0.47,0.05,1,0,0,0,0.00,-1.0,-0.80,0
69160,2,0,2,111.30,143.45,-32.15,0,11.76,0.65,4.94,0.59,31.69,1,0,1,1,0.00,-1.0,0.55,1
69161,0,1,-2,-19.50,4.74,-24.24,0,14.64,-0.01,3.40,0.49,-15.72,0,0,0,0,0.30,-1.0,-0.80,-1
69162,0,1,2,59.47,59.63,-0.16,0,15.33,0.78,2.56,0.73,7.41,0,0,1,1,-0.50,1.0,-0.80,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94308,2,0,2,6.07,4.13,1.94,0,9.57,0.96,3.17,0.51,-8.96,0,0,1,1,0.50,1.0,0.80,-1
94309,1,0,-2,14.50,22.62,-8.12,0,10.66,0.47,2.93,0.14,3.73,1,0,0,0,-0.75,-1.0,0.55,1
94310,2,0,2,4.05,8.06,-4.01,0,4.20,0.28,2.55,0.37,9.63,1,0,0,1,0.85,-0.9,0.80,0
94311,1,0,-2,2.30,2.21,0.08,0,18.18,0.31,4.14,2.05,-17.62,0,0,0,0,0.00,-1.0,-0.80,-1


In [85]:
test_df.to_csv("./data/test_data.csv"  ,index=False)

In [86]:
train_df.columns


Index(['grade', 'model_win', 'RSI_signal', 'MACD', 'MACD_signal', 'MACD_hist',
       'MACD_cross', 'BB_width', 'BB_pct', 'ATR_pct', 'Vol_ratio',
       'Dist_from_200MA', 'sma50_above_sma200', 'perfect_bullish_alignment',
       'price_above_ema9', 'price_above_sma50', 'rev_pat', 'cont_pat',
       'candle_pat', 'dow_signal_encoded'],
      dtype='object')